# 🧠 Multimodal Meme Hate Detection
Image + Text fusion using ResNet50 + Bi-GRU

In [1]:
!pip install kaggle -q

In [2]:
from google.colab import files

files.upload()  # upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"moustafasayed068","key":"2acbd401f144089026d79d063f145819"}'}

In [3]:
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d parthplc/facebook-hateful-meme-dataset -q

Dataset URL: https://www.kaggle.com/datasets/parthplc/facebook-hateful-meme-dataset
License(s): unknown


In [5]:
!unzip -q facebook-hateful-meme-dataset.zip

In [6]:
!git clone https://github.com/mennasherif14/multimodal-meme-analyzer.git

Cloning into 'multimodal-meme-analyzer'...
remote: Enumerating objects: 1009, done.
remote: Counting objects: 100% (1009/1009), done.
remote: Compressing objects: 100% (1008/1008), done.
remote: Total 1009 (delta 1), reused 1009 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (1009/1009), 40.60 MiB | 33.31 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [7]:
import pandas as pd, os

from google.colab import files

files.upload()  # upload merged_dataset.csv

merged = pd.read_csv("merged_dataset.csv")

Saving merged_dataset.csv to merged_dataset.csv


In [8]:
def fix_path(p):

    filename = os.path.basename(p)

    if "9gag" in p or "multimodal" in p:

        return f"/content/multimodal-meme-analyzer/9gag_memes/{filename}"

    else:

        return f"/content/data/img/{filename}"

merged["img"] = merged["img"].apply(fix_path)

## 1. Imports & Setup

In [9]:
import os, re, json, random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Device:', DEVICE)

Device: cuda


## 2. Load & Validate Dataset

In [10]:
df = pd.read_csv('merged_dataset.csv')

# Validate images exist
missing = df['img'].apply(lambda p: not os.path.exists(p)).sum()
print(f'Shape: {df.shape}')
print(f'Missing images: {missing}')  # should be 0
print(df['label'].value_counts())

assert missing == 0, f'{missing} image paths are broken!'

Shape: (9487, 5)
Missing images: 0
label
0    6330
1    3157
Name: count, dtype: int64


## 3. Train / Validation Split

In [11]:
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print(train_df['label'].value_counts())

Train: 7589 | Val: 1898
label
0    5064
1    2525
Name: count, dtype: int64


## 4. Vocabulary

In [12]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

class Vocab:
    def __init__(self, max_size=10000):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.counter  = Counter()
        self.max_size = max_size

    def build(self, texts):
        for t in texts:
            self.counter.update(tokenize(t))
        for w, _ in self.counter.most_common(self.max_size - 2):
            self.word2idx[w] = len(self.word2idx)

    def encode(self, text, max_len=50):
        tokens = tokenize(text)[:max_len]
        ids    = [self.word2idx.get(t, 1) for t in tokens]
        return ids + [0] * (max_len - len(ids))

vocab = Vocab()
vocab.build(train_df['text'])
print('Vocab size:', len(vocab.word2idx))

Vocab size: 10000


## 5. Dataset & DataLoaders

In [13]:
IMG_SIZE = 224
MAX_LEN  = 50

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

class MemeDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = transform(Image.open(row['img']).convert('RGB'))
        text  = torch.tensor(vocab.encode(row['text'], MAX_LEN))
        label = torch.tensor(row['label'], dtype=torch.float32)
        return img, text, label

train_loader = DataLoader(MemeDataset(train_df), batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(MemeDataset(val_df),   batch_size=32, shuffle=False, num_workers=2)
print('Loaders ready.')

Loaders ready.


## 6. Model

In [14]:
class MemeModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # CNN backbone (ResNet50)
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.cnn    = nn.Sequential(*list(resnet.children())[:-1])
        self.img_fc = nn.Linear(2048, 256)

        # Text encoder (Bi-GRU)
        self.embed = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.gru   = nn.GRU(128, 128, batch_first=True, bidirectional=True)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(256 + 256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, img, text):
        x1 = self.img_fc(self.cnn(img).flatten(1))             # (B, 256)
        _, h = self.gru(self.embed(text))                       # h: (2, B, 128)
        x2 = torch.cat([h[-2], h[-1]], dim=1)                  # (B, 256)
        return self.classifier(torch.cat([x1, x2], dim=1)).squeeze(1)

model     = MemeModel(len(vocab.word2idx)).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 142MB/s]


MemeModel(
  (cnn): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64, 25

## 7. Training

In [ ]:
def train_epoch():
    model.train()
    total_loss = 0
    for img, text, y in train_loader:
        img, text, y = img.to(DEVICE), text.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(img, text), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def eval_epoch(threshold=0.4):
    model.eval()
    preds, labels = [], []
    for img, text, y in val_loader:
        prob = torch.sigmoid(model(img.to(DEVICE), text.to(DEVICE))).cpu().numpy()
        preds.extend((prob > threshold).astype(int))
        labels.extend(y.numpy())
    return f1_score(labels, preds), preds, labels

best_f1, patience, counter = 0, 3, 0

for epoch in range(50):
    loss        = train_epoch()
    f1, _, _    = eval_epoch()
    print(f'Epoch {epoch+1:02d} | Loss: {loss:.4f} | F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), 'best.pt')
        print('  ✅ Saved best model')
    else:
        counter += 1
        print(f'  No improvement {counter}/{patience}')
        if counter >= patience:
            print('Early stopping triggered.')
            break

## 8. Evaluation on Validation Set

In [15]:
model.load_state_dict(torch.load('best.pt'))

f1, preds, labels = eval_epoch(threshold=0.4)
print(f'Accuracy : {accuracy_score(labels, preds):.4f}')
print(f'F1 Score : {f1:.4f}')
print('\nConfusion Matrix:\n', confusion_matrix(labels, preds))
print('\nClassification Report:\n', classification_report(labels, preds))

FileNotFoundError: [Errno 2] No such file or directory: 'best.pt'

## 9. OCR — Improved Text Extraction

Standard Tesseract OCR performs poorly on memes due to:
- Impact/bold fonts on complex backgrounds
- Low contrast between text and image
- Noisy or colorful backgrounds

This pipeline applies adaptive preprocessing before OCR and tries multiple configs, keeping the best result.

In [ ]:
!apt-get install -y tesseract-ocr tesseract-ocr-eng
!pip install pytesseract opencv-python-headless

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [17]:
from google.colab import files
uploaded = files.upload()

Saving best.pt to best.pt


In [ ]:
import cv2
import pytesseract

def preprocess_for_ocr(img_path):
    """Return a list of preprocessed images to try OCR on."""
    img_bgr  = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    candidates = []

    # 1. Raw grayscale
    candidates.append(gray)

    # 2. Upscale 2x (helps small fonts)
    h, w  = gray.shape
    large = cv2.resize(gray, (w*2, h*2), interpolation=cv2.INTER_CUBIC)
    candidates.append(large)

    # 3. Adaptive threshold (handles uneven background)
    blurred = cv2.GaussianBlur(large, (3, 3), 0)
    adaptive = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 10
    )
    candidates.append(adaptive)

    # 4. Inverted adaptive (white-on-dark text common in memes)
    candidates.append(cv2.bitwise_not(adaptive))

    # 5. CLAHE — boosts local contrast
    clahe  = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced = clahe.apply(large)
    _, otsu = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    candidates.append(otsu)

    # 6. Morphological dilation to thicken thin text
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    dilated  = cv2.dilate(otsu, kernel, iterations=1)
    candidates.append(dilated)

    return candidates


def ocr_best(img_path, debug=False):
    """
    Run Tesseract with multiple preprocessing strategies and PSM configs.
    Returns the extraction with the most characters (best attempt).
    """
    configs = [
        '--oem 3 --psm 6',   # Assume uniform block of text
        '--oem 3 --psm 11',  # Sparse text — no order assumed
        '--oem 3 --psm 3',   # Fully automatic page segmentation
    ]

    preprocessed = preprocess_for_ocr(img_path)
    best_text, best_len = '', 0

    for proc_img in preprocessed:
        pil_img = Image.fromarray(proc_img)
        for cfg in configs:
            try:
                text = pytesseract.image_to_string(pil_img, config=cfg).strip()
                if debug:
                    print(f'[cfg={cfg}] ({len(text)} chars): {text[:80]}')
                if len(text) > best_len:
                    best_len  = len(text)
                    best_text = text
            except Exception as e:
                if debug: print('Error:', e)

    # Clean up common OCR noise
    best_text = re.sub(r'[^\w\s\'\"\!\?\.,]', ' ', best_text)
    best_text = re.sub(r'\s+', ' ', best_text).strip()

    return best_text


# Quick test
sample = train_df.sample(1).iloc[0]
print('True text :', sample['text'])
print('OCR text  :', ocr_best(sample['img']))

## 10. Predict on User-Uploaded Image

In [ ]:
def predict(img_path, text=None, threshold=0.4, debug_ocr=False):
    """
    Predict whether a meme is hateful.
    - If text is None, OCR is run automatically.
    - Returns (probability, label)
    """
    if not os.path.exists(img_path):
        print('❌ Image not found:', img_path)
        return None

    # Auto-extract text if not provided
    if text is None:
        print('Running OCR...')
        text = ocr_best(img_path, debug=debug_ocr)
        print('Extracted text:', repr(text))

    img    = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    tokens = torch.tensor(vocab.encode(text, MAX_LEN)).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model(img, tokens)).item()

    label = int(prob > threshold)
    print(f'Probability : {prob:.4f}')
    print(f'Prediction  : {"HATEFUL 🚨" if label else "NOT HATEFUL ✅"}')
    return prob, label


# --- Upload & test ---
from google.colab import files

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

predict(img_path, text=None, debug_ocr=True)